# Tutorial 3b — MVPA: Multi-Voxel Pattern Analysis

**Prerequisites:** Run `tutorial_03a_mask_warping.ipynb` first to produce the
per-run warped masks and the common intersection mask.

**Learning objectives**
1. Load a beta-weight matrix and apply ANOVA-based feature selection
2. Train and evaluate an SVM classifier with leave-one-run-out cross-validation
3. Interpret confusion matrices and per-condition accuracy
4. Compute AUC as a threshold-independent discriminability measure
5. Visualise SVM importance maps to identify decodable brain regions

---


## Section 0 — Setup

In [5]:
import os, sys, warnings, glob, re
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import image, plotting
from nilearn.maskers import NiftiMasker
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import (LeaveOneGroupOut, StratifiedKFold,
                                     cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_auc_score)
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from fmri_helpers import (
    load_beta_matrix, run_svm_decoding,
    compute_importance_map, plot_confusion_matrix, track_runtime,
)

# ── Subject / paths ───────────────────────────────────────────────────────────
SUBJECT    = "sub-1"
USER       = os.getenv("USER")
DERIV_ROOT = f"/scratch/alpine/{USER}/mvpa_workshop/ds000105/derivatives"
N_RUNS     = 12

BETA_DIR   = f"{DERIV_ROOT}/beta_weights/{SUBJECT}"
WARPED_DIR = f"{BETA_DIR}/warped_masks"
OUTPUT_DIR = f"{DERIV_ROOT}/mvpa/all_conditions_lss/{SUBJECT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Common intersection mask (produced by tutorial_03a_mask_warping.ipynb) ───
MASK_FILE = (
    f"{WARPED_DIR}/{SUBJECT}_task-objectviewing"
    f"_desc-warpedGroupMaskCommon_funcspace.nii.gz"
)

# ── Functional reference (for visualisation) ──────────────────────────────────
func_ref_path = (
    f"{DERIV_ROOT}/fmriprep/{SUBJECT}/func/"
    f"{SUBJECT}_task-objectviewing_run-01_boldref.nii.gz"
)

print(f"Mask exists     : {os.path.exists(MASK_FILE)}")
if not os.path.exists(MASK_FILE):
    print("\n⚠️  Mask not found — run tutorial_03a_mask_warping.ipynb first!")


Mask exists     : True


---
## Section 1 — What is MVPA?

Classical fMRI analysis asks: *"Is this region more active during condition A than B?"*
(univariate, mass-univariate, or ROI mean comparison).

**MVPA** asks instead: *"Does the spatial pattern of activation across this region
contain information that distinguishes condition A from B?"*

This is a fundamentally different and often more sensitive question because:
- A region can carry **distributed information** without a detectable mean change
- The classifier exploits the **co-variance structure** across voxels
- Even voxels with sub-threshold individual activation can contribute to decoding

### The pipeline

```
Beta images (1 per trial)
    → Feature matrix X  (n_trials × n_voxels)
    → [Optional] Feature selection  (e.g. ANOVA-F to keep top voxels)
    → SVM classifier
    → Leave-one-run-out cross-validation
    → Decoding accuracy, confusion matrix, importance map
```

### Support Vector Machine (SVM)

The SVM finds a **maximum-margin hyperplane** that separates classes in the
feature space. With a linear kernel, the decision boundary is:

$$f(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b$$

The weight vector $\mathbf{w}$ has the same dimensionality as the feature space
(= number of voxels), and its magnitude at each voxel indicates the voxel's
contribution to discrimination — our **importance map**.


---
## Section 2 — Loading Beta Weights and Building the Feature Matrix


In [11]:
# ── Step 1: Discover beta manifests across all runs ──────────────────────────
# Betas may be stored in per-run subdirectories OR in a single combined CSV.
# We check both patterns, prefer the one that covers more runs.

def _load_manifests(beta_dir, n_runs, method='lss'):
    "Return a combined DataFrame from per-run or flat manifests."
    # Pattern A: per-run subdirectories  → beta_weights/sub-1/run-01/beta_manifest_lss.csv
    per_run = sorted(glob.glob(f"{beta_dir}/run-*/beta_manifest_{method}.csv"))
    # Pattern B: flat combined manifest  → beta_weights/sub-1/beta_manifest_lss.csv
    flat    = f"{beta_dir}/beta_manifest_{method}.csv"

    if per_run:
        dfs = []
        for csv_path in per_run:
            m = re.search(r'run-0*(\d+)', csv_path)
            run_id = int(m.group(1)) if m else None
            df = pd.read_csv(csv_path)
            # Ensure a numeric 'run' column
            if run_id is not None:
                df['run'] = run_id
            dfs.append(df)
        combined = pd.concat(dfs, ignore_index=True)
        print(f"  Loaded {len(per_run)} per-run manifest(s) from {beta_dir}/run-*/")
    elif os.path.exists(flat):
        combined = pd.read_csv(flat)
        print(f"  Loaded combined manifest: {flat}")
    else:
        raise FileNotFoundError(
            f"No beta manifests found in {beta_dir}.\n"
            f"Expected either {flat} or {beta_dir}/run-XX/beta_manifest_{method}.csv"
        )
    return combined

manifest = _load_manifests(BETA_DIR, N_RUNS, method='lss')

# ── Step 2: Diagnostic — confirm all runs are present ────────────────────────
run_col = next((c for c in ['run', 'run_id', 'run_num'] if c in manifest.columns), None)
if run_col is None:
    raise KeyError(
        f"No run column found in manifest. Columns: {list(manifest.columns)}")

run_counts = manifest.groupby(run_col).size().sort_index()
print(f"\nTrials per run:")
for run_id, n in run_counts.items():
    print(f"  Run {str(run_id):>4s}: {n:3d} trials")

n_unique = manifest[run_col].nunique()
print(f"\nTotal : {len(manifest)} trials across {n_unique} run(s)")

if n_unique < 2:
    print("\n⚠️  Only 1 run detected — LORO cross-validation requires ≥ 2 runs.")
    print("   Check that beta estimation was run for all runs.")

# ── Step 3: Save combined manifest (load_beta_matrix expects a CSV path) ─────
combined_csv = os.path.join(BETA_DIR, "beta_manifest_lss_combined.csv")
manifest.to_csv(combined_csv, index=False)

# ── Step 4: Load feature matrix using the common intersection mask ────────────
mask_img = image.load_img(MASK_FILE)

X, y_raw, runs_arr, masker, active_voxels, n_mask_voxels = load_beta_matrix(
    combined_csv, mask_img=mask_img)

le = LabelEncoder()
y  = le.fit_transform(y_raw)

conditions = list(le.classes_)
print(f"\nConditions  : {conditions}")
print(f"X shape     : {X.shape}  (n_trials × n_active_voxels)")
print(f"Unique runs : {np.unique(runs_arr)}  ({len(np.unique(runs_arr))} runs)")
print(f"Chance level: {100/len(conditions):.1f}%")
print(f"Trials per condition:\n{pd.Series(y_raw).value_counts().sort_index()}")


  Loaded combined manifest: /scratch/alpine/amhe4269/mvpa_workshop/ds000105/derivatives/beta_weights/sub-1/beta_manifest_lss.csv

Trials per run:
  Run    1:  96 trials

Total : 96 trials across 1 run(s)

⚠️  Only 1 run detected — LORO cross-validation requires ≥ 2 runs.
   Check that beta estimation was run for all runs.
  Beta matrix: 96 trials × 6152 active voxels  (100.0% of 6152 mask voxels)

Conditions  : [np.str_('bottle'), np.str_('cat'), np.str_('chair'), np.str_('face'), np.str_('house'), np.str_('scissors'), np.str_('scrambledpix'), np.str_('shoe')]
X shape     : (96, 6152)  (n_trials × n_active_voxels)
Unique runs : [1]  (1 runs)
Chance level: 12.5%
Trials per condition:
bottle          12
cat             12
chair           12
face            12
house           12
scissors        12
scrambledpix    12
shoe            12
Name: count, dtype: int64


### ✏️ Exercise 1 — Explore the beta weight matrix

1. Print the mean and standard deviation of the entire feature matrix X
2. Plot a histogram of all values in X
3. Compute the mean beta value per condition (across all trials and voxels)
4. Do conditions differ in mean activation level?


In [ ]:
# YOUR CODE HERE
print(f"X mean : {X.mean():.4f}")
print(f"X std  : {X.std():.4f}")

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
# ... your code

# Mean per condition
for i, cond in enumerate(le.classes_):
    mask_cond = (y == i)
    # ... your code


---
## Section 3 — Feature Selection with ANOVA

With thousands of voxels, many contain no condition-relevant signal —
they are pure noise. Including them can hurt SVM performance (curse of
dimensionality) and slow training.

**ANOVA F-score feature selection:**
For each voxel, compute the one-way ANOVA F-statistic across conditions
(how much does the mean beta differ across conditions relative to within-condition variance?).
Keep the top N% of voxels.

This is done **within each training fold** in sklearn Pipelines to avoid data leakage.


In [ ]:
from scipy.stats import f_oneway

# Compute per-voxel ANOVA F-scores on the full dataset (for visualisation only)
# In practice, SelectPercentile() inside a Pipeline does this properly (within fold)
conditions_idx = [np.where(y == i)[0] for i in range(len(conditions))]
groups_per_vox = [[X[idx, v] for idx in conditions_idx] for v in range(X.shape[1])]

# This is slow for large X — we'll use scikit-learn's SelectPercentile instead
selector = SelectPercentile(f_classif, percentile=10)
selector.fit(X, y)

F_scores   = selector.scores_          # one F-score per voxel
selected   = selector.get_support()   # boolean, True = kept
n_selected = int(selected.sum())

print(f"Voxels in mask         : {X.shape[1]}")
print(f"Voxels selected (top 10%): {n_selected}")

# Map F-scores to NIfTI for visualisation
full_f    = np.zeros(n_mask_voxels)
full_f[active_voxels] = F_scores
F_img     = masker.inverse_transform(full_f)
F_img.to_filename(os.path.join(OUTPUT_DIR, f"{SUBJECT}_anova_f_scores.nii.gz"))

# Also make selected-voxel mask image
full_sel    = np.zeros(n_mask_voxels)
full_sel[active_voxels] = selected.astype(float)
sel_img     = masker.inverse_transform(full_sel)

func_ref_path = (f"/pl/active/courses/2026_summer/neuroclass2026/"
                  f"ds000105/derivatives/fmriprep/{SUBJECT}/func/"
                  f"{SUBJECT}_task-objectviewing_run-1_boldref.nii.gz")
func_bg = nib.load(func_ref_path) if os.path.exists(func_ref_path) else None

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plotting.plot_stat_map(F_img, bg_img=func_bg, display_mode='z', cut_coords=7,
                       colorbar=True, title='ANOVA F-scores (all voxels)',
                       cmap='hot', axes=axes[0])
plotting.plot_roi(sel_img, bg_img=func_bg if func_bg else F_img,
                  display_mode='z', cut_coords=7,
                  title=f'ANOVA-selected voxels (top 10%, n={n_selected})',
                  cmap='Blues', alpha=0.6, axes=axes[1])
plt.tight_layout(); plt.show()


---
## Section 4 — Cross-Validation Strategy

We use **leave-one-run-out (LORO) cross-validation**:
- Train on 11 runs → test on the left-out run
- Repeat 12 times (once per run)
- Report mean ± SEM accuracy across folds

This is preferred over k-fold for task fMRI because:
1. Each fold's test set is temporally independent of training (no autocorrelation leakage)
2. It mirrors the natural experiment structure
3. Results are easier to interpret (accuracy per run)

The full sklearn Pipeline is:
```
SelectPercentile(f_classif, percentile=10)  →  SVC(kernel='linear', C=1)
```


In [ ]:
# Run SVM decoding using fmri_helpers convenience function
with track_runtime("cross-validation"):
    results = run_svm_decoding(
        X                  = X,
        y_raw              = y_raw,
        runs_arr           = runs_arr,
        feature_selection  = 'anova',
        anova_percentile   = 10.0,
        svm_C              = 1.0,
        svm_kernel         = 'linear',
        leave_one_run_out  = True,
    )

print(f"\nMean accuracy   : {results['mean_acc']*100:.2f}% ± {results['sem_acc']*100:.2f}% SEM")
print(f"Balanced acc    : {balanced_accuracy_score(results['y'], results['y_pred'])*100:.2f}%")
print(f"Chance level    : {100/len(conditions):.1f}%")
print(f"\nPer-fold accuracies:")
for i, acc in enumerate(results['fold_accs']):
    print(f"  Run {i+1:02d}: {acc*100:.1f}%")


### ✏️ Exercise 2 — Manual cross-validation loop

Implement the leave-one-run-out cross-validation loop manually (without
using `cross_val_predict`) to see what happens at each fold:

1. For each run as test set:
   - Split X and y into training and test sets
   - Fit `SelectPercentile(f_classif, percentile=10)` on training data only
   - Fit `SVC(kernel='linear', C=1)` on selected training features
   - Predict on test set features
   - Compute accuracy
2. Print accuracy for each fold
3. Compute and print the mean accuracy

**Important:** Feature selection must be fit ONLY on training data.
Fitting on all data would constitute data leakage.


In [ ]:
# YOUR CODE HERE
fold_accs_manual = []
unique_runs = np.unique(runs_arr)

for test_run in unique_runs:
    # Split
    train_idx = np.where(runs_arr != test_run)[0]
    test_idx  = np.where(runs_arr == test_run)[0]

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = results['y'][train_idx], results['y'][test_idx]

    # YOUR CODE: feature selection + classifier

    # Hint:
    # sel = SelectPercentile(...)
    # sel.fit(X_train, y_train)
    # X_train_sel = sel.transform(X_train)
    # X_test_sel  = sel.transform(X_test)
    # clf = SVC(...)
    # clf.fit(X_train_sel, y_train)
    # y_pred_fold = clf.predict(X_test_sel)
    # acc = accuracy_score(y_test, y_pred_fold)

    fold_accs_manual.append(acc)
    print(f"  Run {test_run:02d} (test): {acc*100:.1f}%")

print(f"\nMean accuracy (manual loop): {np.mean(fold_accs_manual)*100:.2f}%")


---
## Section 5 — Interpreting Results: Confusion Matrix and Per-Condition Accuracy


In [ ]:
print(classification_report(results['y'], results['y_pred'], target_names=le.classes_))

fig = plot_confusion_matrix(
    results['cm'], results['cm_raw'], list(le.classes_), title_prefix=SUBJECT)
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# Per-condition bar chart
per_cond = {c: results['cm'][i, i] for i, c in enumerate(le.classes_)}
fig, ax  = plt.subplots(figsize=(max(5, len(conditions)*1.2), 4))
bars = ax.bar(per_cond.keys(), [v*100 for v in per_cond.values()],
              color='steelblue', edgecolor='k', alpha=0.85)
ax.axhline(100/len(conditions), color='gray', lw=1.2, ls='--', label='Chance')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)'); ax.legend()
ax.set_title(f'Per-condition accuracy — mean: {results["mean_acc"]*100:.2f}%')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()


### ✏️ Exercise 3 — AUC analysis

The confusion matrix shows accuracy, but **AUC (area under the ROC curve)**
is a threshold-independent measure of discriminability.

Using `roc_auc_score` with the cross-validated class probabilities (`y_proba`):
1. Compute the per-condition AUC using the one-vs-rest approach
2. Create a bar chart of AUC values per condition
3. Which condition has the highest AUC? Does this match the confusion matrix?

**Hint:** `roc_auc_score(y, y_proba, multi_class='ovr', average=None)` returns
an AUC value for each class.


In [ ]:
# YOUR CODE HERE
from sklearn.metrics import roc_auc_score

auc_values = roc_auc_score(
    results['y'],
    results['y_proba'],
    multi_class = ...,
    average     = ...,
)

print("AUC per condition:")
for cond, auc in zip(le.classes_, auc_values):
    print(f"  {cond:<20} AUC = {auc:.3f}")

# Bar chart
# ...


---
## Section 6 — SVM Importance Maps

The SVM's linear weight vector $\mathbf{w}$ directly encodes which voxels
drive the classification decision. Large $|w_i|$ at voxel $i$ means that voxel
contributed strongly to at least one discriminative hyperplane.

For multi-class OvO (one-vs-one) SVMs, there is one hyperplane per pair of conditions.
We average $|w|$ across all hyperplanes to get a single summary importance map.


In [ ]:
imp_img = compute_importance_map(
    pipe          = results['pipe'],
    X             = X,
    masker        = masker,
    active_voxels = active_voxels,
    n_mask_voxels = n_mask_voxels,
)

imp_path = os.path.join(OUTPUT_DIR, f"{SUBJECT}_importance_map_funcspace.nii.gz")
imp_img.to_filename(imp_path)
print(f"Saved: {imp_path}")

imp_data = imp_img.get_fdata()
func_bg = nib.load(func_ref_path) if os.path.exists(func_ref_path) else None

fig, ax = plt.subplots(figsize=(14, 4))
plotting.plot_stat_map(
    imp_img, bg_img=func_bg,
    display_mode='z', cut_coords=7,
    colorbar=True, cmap='hot',
    threshold=np.percentile(imp_data[imp_data > 0], 50),
    title='SVM importance map — mean |w| across hyperplanes',
    axes=ax,
)
plt.tight_layout(); plt.show()


---
## Summary

| Concept | Key take-away |
|---|---|
| Feature matrix | Stack of beta images: n_trials × n_voxels |
| ANOVA selection | Retain voxels with significant condition differences (within fold!) |
| SVM + LORO-CV | Leave-one-run-out ensures temporally clean train/test splits |
| Confusion matrix | Off-diagonal entries show systematic confusions between conditions |
| Importance map | Mean \|SVM weight\| per voxel — highlights decodable regions |

---


---
## 📖 Answer Key

### Exercise 1 — Explore the beta weight matrix

In [ ]:
# ANSWER KEY — Exercise 1
print(f"X mean : {X.mean():.4f}")
print(f"X std  : {X.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(X.ravel(), bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0, color='red', lw=1.2, ls='--')
ax.set_xlabel('Beta weight'); ax.set_ylabel('Count')
ax.set_title('Distribution of all voxel × trial beta values'); plt.tight_layout(); plt.show()

print("\nMean beta per condition:")
for i, cond in enumerate(le.classes_):
    mean_val = X[y == i].mean()
    print(f"  {cond:<20} mean = {mean_val:.4f}")
# Differences are small because mean activation across the whole brain
# is similar; MVPA exploits SPATIAL PATTERNS, not overall mean level.


### Exercise 2 — Manual cross-validation loop

In [ ]:
# ANSWER KEY — Exercise 2
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

fold_accs_manual = []
unique_runs = np.unique(runs_arr)

for test_run in unique_runs:
    train_idx = np.where(runs_arr != test_run)[0]
    test_idx  = np.where(runs_arr == test_run)[0]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = results['y'][train_idx], results['y'][test_idx]

    sel = SelectPercentile(f_classif, percentile=10)
    sel.fit(X_train, y_train)                 # fit on TRAIN only — no leakage!
    X_train_sel = sel.transform(X_train)
    X_test_sel  = sel.transform(X_test)

    clf = SVC(C=1.0, kernel='linear', class_weight='balanced', random_state=42)
    clf.fit(X_train_sel, y_train)
    y_pred_fold = clf.predict(X_test_sel)
    acc = accuracy_score(y_test, y_pred_fold)

    fold_accs_manual.append(acc)
    print(f"  Run {test_run:02d}: {acc*100:.1f}%")

print(f"\nMean accuracy: {np.mean(fold_accs_manual)*100:.2f}%")


### Exercise 3 — AUC analysis

In [ ]:
# ANSWER KEY — Exercise 3
auc_values = roc_auc_score(results['y'], results['y_proba'],
                            multi_class='ovr', average=None)
print("AUC per condition (OvR):")
for cond, auc in sorted(zip(le.classes_, auc_values), key=lambda x: -x[1]):
    print(f"  {cond:<20} AUC = {auc:.3f}")

fig, ax = plt.subplots(figsize=(max(5, len(conditions)*1.2), 4))
ax.bar(le.classes_, auc_values, color='teal', edgecolor='k', alpha=0.85)
ax.axhline(0.5, color='gray', lw=1.2, ls='--', label='Chance AUC=0.5')
ax.bar_label(ax.containers[0], fmt='%.3f', padding=3, fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('AUC (OvR)'); ax.legend()
ax.set_title('Per-condition AUC — one-vs-rest')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
